In [ ]:
import os
import time
import requests
import pandas as pd
from tqdm.auto import tqdm

# ==========================================\n
# 0. CONFIGURACIÓN GLOBAL Y RUTAS (Estilo main.ipynb)
# ==========================================\n
TOKEN_BANXICO = "3cf05ba180ebc8fa6bac83d6473f5c287fd4a5d28c8d0411ec9c2e896b844b3e"

try:
    SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
    if os.path.basename(SCRIPT_DIR) == 'scripts':
        PROJECT_ROOT = os.path.dirname(SCRIPT_DIR)
    else:
        PROJECT_ROOT = SCRIPT_DIR
except NameError:
    PROJECT_ROOT = os.getcwd()
    if os.path.basename(PROJECT_ROOT) == 'scripts':
        PROJECT_ROOT = os.path.dirname(PROJECT_ROOT)

INTERMEDIATE_DIR = os.path.join(PROJECT_ROOT, "data", "intermediate")
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)

# ==========================================\n
# DICCIONARIO DE CONSULTA
# ==========================================\n
entidades_banxico = {
    "SE28528": "remesas totales trimestrales",
    "SE29670": "Aguascalientes",
    "SE29671": "Baja California",
    "SE29672": "Baja California Sur",
    "SE29673": "Campeche",
    "SE29674": "Coahuila",
    "SE29675": "Colima",
    "SE29676": "Chiapas",
    "SE29677": "Chihuahua",
    "SE29678": "Ciudad de México",
    "SE29679": "Durango",
    "SE29680": "Guanajuato",
    "SE29681": "Guerrero",
    "SE29682": "Hidalgo",
    "SE29683": "Jalisco",
    "SE29684": "México",
    "SE29685": "Michoacán",
    "SE29686": "Morelos",
    "SE29687": "Nayarit",
    "SE29688": "Nuevo León",
    "SE29689": "Oaxaca",
    "SE29690": "Puebla",
    "SE29691": "Querétaro",
    "SE29692": "Quintana Roo",
    "SE29693": "San Luis Potosí",
    "SE29694": "Sinaloa",
    "SE29695": "Sonora",
    "SE29696": "Tabasco",
    "SE29697": "Tamaulipas",
    "SE29698": "Tlaxcala",
    "SE29699": "Veracruz",
    "SE29700": "Yucatán",
    "SE29701": "Zacatecas"
}

# ==========================================\n
# 1. FUNCIÓN CORE DE EXTRACCIÓN (Adaptada)
# ==========================================\n
def obtener_serie_banxico(token, serie_id):
    """
    Descarga datos de una serie de Banxico y los regresa como DataFrame.
    Incorpora manejo de errores básico para tolerancia a fallos de red.
    """
    url = f"https://www.banxico.org.mx/SieAPIRest/service/v1/series/{serie_id}/datos"
    headers = {"Bmx-Token": token}
    
    for intento in range(3):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 200:
                data = response.json()
                serie_info = data['bmx']['series'][0]
                titulo = serie_info['titulo']
                datos = serie_info['datos']
                
                df = pd.DataFrame(datos)
                # Renombrar columnas a un estándar
                df.columns = ['fecha', titulo]
                df['fecha'] = pd.to_datetime(df['fecha'], format='%d/%m/%Y')
                df[titulo] = pd.to_numeric(df[titulo].str.replace(',', ''), errors='coerce')
                return df
            else:
                time.sleep(1) # Pausa antes de reintentar si hay error de servidor
        except requests.exceptions.RequestException:
            time.sleep(1)
            
    # Si falla después de 3 intentos, devuelve DataFrame vacío
    return pd.DataFrame(columns=['fecha', 'valor_nulo'])

# ==========================================\n
# 2. FUNCIÓN ORQUESTADORA (Lista para main.ipynb)
# ==========================================\n
def procesar_remesas_banxico():
    print("⏳ [Remesas Banxico] Iniciando extracción trimestral...")
    
    # Base de fechas desde 2003 (inicio de la serie en Banxico para este bloque)
    dates = pd.date_range(start="2003-01-01", end=pd.Timestamp.today(), freq="QS")
    df_master = pd.DataFrame({"fecha": dates})
    
    errores = 0
    
    # Barra de progreso al estilo de tu ETL principal
    barra_progreso = tqdm(entidades_banxico.items(), desc="💸 Remesas     ", unit="estado")
    
    for serie_id, nombre_columna in barra_progreso:
        df_temp = obtener_serie_banxico(TOKEN_BANXICO, serie_id)
        
        if not df_temp.empty and len(df_temp.columns) == 2:
            # Renombramos la columna al nombre del estado/rubro del diccionario
            df_temp.columns = ['fecha', nombre_columna]
            df_master = df_master.merge(df_temp, on="fecha", how="left")
        else:
            errores += 1
            barra_progreso.set_postfix({"Errores": errores})
            
    if not df_master.empty:
        # Formateo final para el CSV
        df_master = df_master.sort_values(by='fecha').reset_index(drop=True)
        # Opcional: convertir fecha a string para estandarizar en CSV
        df_master['fecha'] = df_master['fecha'].dt.strftime('%Y-%m-%d')
        
        ruta_salida = os.path.join(INTERMEDIATE_DIR, "remesas_entidad.csv")
        df_master.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
        
        return f"✅ [Remesas Banxico] Completado ({len(df_master)} periodos). Guardado en {ruta_salida}"
    else:
        return "⚠️ [Remesas Banxico] No se pudieron extraer los datos."

if __name__ == "__main__":
    resultado = procesar_remesas_banxico()
    print(f"\n{resultado}")

⏳ [Remesas Banxico] Iniciando extracción trimestral...


💸 Remesas     :   0%|          | 0/33 [00:00<?, ?estado/s]


✅ [Remesas Banxico] Completado (94 periodos). Guardado en c:\Users\e_edobru\Desktop\Bancomext\Estatales\data\intermediate\remesas_trimestral_entidades.csv
